In [1]:
!pip install pandas

In [2]:
#installing necessary packages
!pip install python-dotenv
!pip install azure-storage-blob

In [3]:
#importing necessary libraries
import pandas as pd
from azure.storage.blob import BlobServiceClient, BlobClient, ContainerClient
from dotenv import load_dotenv
import os

In [4]:
# Data Extraction
try:
    data = pd.read_csv(r'zipco_transaction.csv')
    print("Data loaded successfully!")
except Exception as e:
    print(f"An error occurred: {e}")

Data loaded successfully!


In [5]:
data.head()

,Date,ProductName,Quantity,UnitPrice,StoreLocation,PaymentType,PromotionApplied,Weather,Temperature,StaffPerformanceRating,...,DeliveryTime_min,OrderType,CustomerName,CustomerAddress,Customer_PhoneNumber,CustomerEmail,Staff_Name,Staff_Email,DayOfWeek,TotalSales
0,2023-01-01 00:00:00,Vanilla Cake,2,12.532304,South,Cash,True,Rainy,20.654914,Poor,...,30,In Store,William Adams,"9851 David Green\r\nTonyaburgh, VA 02853",(916)427-7276x861,lisa00@example.net,John Bridges,pdavidson@example.com,Sunday,25.064608
1,2023-01-01 01:00:00,Red Velvet Cake,1,7.083070,South,Cash,False,Rainy,23.549497,Average,...,33,In Store,Anthony Wiggins,"24682 Holly Stravenue\r\nMooreville, NH 13901",769.318.4373,michellefernandez@example.com,Sarah Bentley,ajohnson@example.net,Sunday,7.083070
2,2023-01-01 02:00:00,Chocolate Cake,5,6.736064,North,Cash,True,Rainy,NaN,Excellent,...,43,Phone Order,Ashley Duke,10184 Washington Trace Apt. 679\r\nEast Brandi...,702.520.3286,cooperwilliam@example.com,Connie Cervantes,michele29@example.net,Sunday,33.680321
3,2023-01-01 03:00:00,Carrot Cake,2,7.314823,North,Cash,False,Cloudy,20.137483,Poor,...,32,Online Order,Brandon Taylor,"87194 Jeff Rue\r\nMitchellbury, CA 50463",622-527-9530,fsilva@example.net,Jessica Stewart,xwilson@example.org,Sunday,14.629647
4,2023-01-01 04:00:00,Pizza Pepperoni,1,7.577727,East,Credit Card,True,Cloudy,23.020987,Good,...,58,In Store,Brittany Watkins,"850 Julia Groves\r\nHartview, WI 95954",759-517-8359,petersjoseph@example.net,Cheryl Carpenter,christine96@example.org,Sunday,7.577727


In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Date                    1005 non-null   str    
 1   ProductName             1005 non-null   str    
 2   Quantity                1005 non-null   int64  
 3   UnitPrice               1005 non-null   float64
 4   StoreLocation           1005 non-null   str    
 5   PaymentType             1005 non-null   str    
 6   PromotionApplied        1005 non-null   bool   
 7   Weather                 1005 non-null   str    
 8   Temperature             904 non-null    float64
 9   StaffPerformanceRating  1005 non-null   str    
 10  CustomerFeedback        905 non-null    str    
 11  DeliveryTime_min        1005 non-null   int64  
 12  OrderType               1005 non-null   str    
 13  CustomerName            1005 non-null   str    
 14  CustomerAddress         1005 non-null   str    
 15

In [7]:
data['Date'] = pd.to_datetime(data['Date'])

In [8]:
# Data cleaning and transformation
# Remove duplicates
data.drop_duplicates(inplace=True)

# Handling missing values (Example: fill missing numeric values with the mean or median)
numeric_columns = data.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_columns:
    #data[col].fillna(data[col].mean(), inplace=True)
    data.fillna({col: data[col].mean()}, inplace=True)

In [9]:
# Handling missing values (Example: fill missing string values with 'Unknown')
string_columns = data.select_dtypes(include=['object']).columns
for col in string_columns:
    #data[col].fillna('Unknown', inplace=True)
    data.fillna({col: 'Unknown'}, inplace=True)

C:\Users\euzok\AppData\Local\Temp\ipykernel_23816\2815759073.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = data.select_dtypes(include=['object']).columns


In [10]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    1000 non-null   datetime64[us]
 1   ProductName             1000 non-null   str           
 2   Quantity                1000 non-null   int64         
 3   UnitPrice               1000 non-null   float64       
 4   StoreLocation           1000 non-null   str           
 5   PaymentType             1000 non-null   str           
 6   PromotionApplied        1000 non-null   bool          
 7   Weather                 1000 non-null   str           
 8   Temperature             1000 non-null   float64       
 9   StaffPerformanceRating  1000 non-null   str           
 10  CustomerFeedback        1000 non-null   str           
 11  DeliveryTime_min        1000 non-null   int64         
 12  OrderType               1000 non-null   str           
 13  

In [11]:
# Create Products Table
products = data[['ProductName', 'UnitPrice']].drop_duplicates().reset_index(drop=True)
products.index.name = 'ProductID'
products = products.reset_index()

In [12]:
print("Products Table:")
print(products.head())


Products Table:
   ProductID      ProductName  UnitPrice
0          0     Vanilla Cake  12.532304
1          1  Red Velvet Cake   7.083070
2          2   Chocolate Cake   6.736064
3          3      Carrot Cake   7.314823
4          4  Pizza Pepperoni   7.577727


In [13]:
# Create Customers Table
customers = data[['CustomerName', 'CustomerAddress', 'Customer_PhoneNumber', 'CustomerEmail']].drop_duplicates().reset_index(drop=True)
customers.index.name = 'CustomerID'
customers = customers.reset_index()

In [14]:
# Create Staff Table
staff = data[['Staff_Name', 'Staff_Email']].drop_duplicates().reset_index(drop=True)
staff.index.name = 'StaffID'
staff = staff.reset_index()

In [15]:
# Create Transaction Table
transactions = data.merge(products, on = ['ProductName', 'UnitPrice'], how='left') \
                   .merge(customers, on = ['CustomerName', 'CustomerAddress', 'Customer_PhoneNumber', 'CustomerEmail'], how='left') \
                   .merge(staff, on= ['Staff_Name', 'Staff_Email'], how='left')
transactions.index.name = 'TransactionID'
transactions = transactions.reset_index() \
                           [['TransactionID', 'Date', 'ProductID', 'CustomerID', 'StaffID', 'Quantity', 'StoreLocation', 'PaymentType', \
                                'PromotionApplied', 'Weather', 'Temperature', 'StaffPerformanceRating', 'CustomerFeedback', \
                                'DeliveryTime_min', 'OrderType', 'DayOfWeek', 'TotalSales']]

In [16]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   TransactionID           1000 non-null   int64         
 1   Date                    1000 non-null   datetime64[us]
 2   ProductID               1000 non-null   int64         
 3   CustomerID              1000 non-null   int64         
 4   StaffID                 1000 non-null   int64         
 5   Quantity                1000 non-null   int64         
 6   StoreLocation           1000 non-null   str           
 7   PaymentType             1000 non-null   str           
 8   PromotionApplied        1000 non-null   bool          
 9   Weather                 1000 non-null   str           
 10  Temperature             1000 non-null   float64       
 11  StaffPerformanceRating  1000 non-null   str           
 12  CustomerFeedback        1000 non-null   str           
 13  

In [17]:
transactions.head()

,TransactionID,Date,ProductID,CustomerID,StaffID,Quantity,StoreLocation,PaymentType,PromotionApplied,Weather,Temperature,StaffPerformanceRating,CustomerFeedback,DeliveryTime_min,OrderType,DayOfWeek,TotalSales
0,0,2023-01-01 00:00:00,0,0,0,2,South,Cash,True,Rainy,20.654914,Poor,Neutral,30,In Store,Sunday,25.064608
1,1,2023-01-01 01:00:00,1,1,1,1,South,Cash,False,Rainy,23.549497,Average,Positive,33,In Store,Sunday,7.083070
2,2,2023-01-01 02:00:00,2,2,2,5,North,Cash,True,Rainy,27.154342,Excellent,Unknown,43,Phone Order,Sunday,33.680321
3,3,2023-01-01 03:00:00,3,3,3,2,North,Cash,False,Cloudy,20.137483,Poor,Positive,32,Online Order,Sunday,14.629647
4,4,2023-01-01 04:00:00,4,4,4,1,East,Credit Card,True,Cloudy,23.020987,Good,Neutral,58,In Store,Sunday,7.577727


In [18]:
# Save normalized tables to new CSV files
data.to_csv('data/clean_data.csv', index=False)
products.to_csv('data/products.csv', index=False)
customers.to_csv('data/customers.csv', index=False)
staff.to_csv('data/staff.csv', index=False)
transactions.to_csv('data/transactions.csv', index=False)

In [19]:
# Azure connection string and container name
# Load environment variables from .env file
load_dotenv()
connect_str = os.getenv('AZURE_CONNECTION_STRING_VALUE')
container_name = os.getenv('CONTAINER_NAME')




In [20]:
load_dotenv(".env")

True

In [21]:
import os
from dotenv import load_dotenv

print("Current folder:", os.getcwd())
print("Files here:", os.listdir())

loaded = load_dotenv()
print("Dotenv loaded:", loaded)

connect_str = os.getenv("AZURE_CONNECTION_STRING_VALUE")
container_name = os.getenv("CONTAINER_NAME")

print("Connection string exists:", connect_str is not None)
print("Container name:", container_name)

Current folder: c:\Users\euzok\OneDrive\Desktop\Zipco_food_Etl_case_study_using_Apache_Airflow_for _orchestration
Files here: ['.env', '.git', '.gitignore', '.venv', 'data', 'etl_pipeline.ipynb', 'zipco_transaction.csv']
Dotenv loaded: True
Connection string exists: True
Container name: zipcofoodscontainer


In [22]:
import os
print(os.getcwd())

c:\Users\euzok\OneDrive\Desktop\Zipco_food_Etl_case_study_using_Apache_Airflow_for _orchestration


In [ ]:
# Create a BlobServiceClient object
blob_service_client = BlobServiceClient.from_connection_string(connect_str)
container_client = blob_service_client.get_container_client(container_name)

# Load data to Azure Blob Storage
# List of tuples (DataFrame, Blob Name)
files = [
    (data, "rawdata/cleaned_zipco_transaction_data.csv"),
    (products, "cleaneddata/products.csv"),
    (customers, "cleaneddata/customers.csv"),
    (staff, "cleaneddata/staff.csv"),
    (transactions, "cleaneddata/transactions.csv")
]

# Load data to Azure Blob Storage
for file, blob_name in files:
    blob_client = container_client.get_blob_client(blob_name)
    output = file.to_csv(index=False)
    blob_client.upload_blob(output, overwrite=True)
    print(f"{blob_name} loaded into Azure Blob Storage.")

rawdata/cleaned_zipco_transaction_data.csv loaded into Azure Blob Storage.
cleaneddata/products.csv loaded into Azure Blob Storage.
cleaneddata/customers.csv loaded into Azure Blob Storage.
cleaneddata/staff.csv loaded into Azure Blob Storage.
cleaneddata/transactions.csv loaded into Azure Blob Storage.
